# Medical Text BERT Fine-Tuning - Reference Solution

Fine-tunes BioBERT (or bert-base-uncased fallback) for 7-class clinical note classification.

In [ ]:
import csv, torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

LABELS = ['cardiovascular','respiratory','neurological','gastrointestinal','musculoskeletal','endocrine','dermatological']
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
def load_data(path):
    with open(path, 'r', encoding='utf-8') as f:
        return list(csv.DictReader(f))

train_rows = load_data('dataset/public/train.csv')
test_rows = load_data('dataset/public/test.csv')
print(f'Train: {len(train_rows)} | Test: {len(test_rows)}')

In [ ]:
MODEL_NAME = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(LABELS)).to(DEVICE)

class NoteDS(Dataset):
    def __init__(self, rows, labeled=True):
        self.rows, self.labeled = rows, labeled
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows[idx]
        enc = tokenizer(row['text'], truncation=True, max_length=256, padding='max_length', return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labeled:
            item['labels'] = torch.tensor(LABEL2ID[row['condition']])
        return item

train_dl = DataLoader(NoteDS(train_rows), batch_size=16, shuffle=True)
optimizer = AdamW(model.parameters(), lr=2e-5)
scheduler = get_linear_schedule_with_warmup(optimizer, 0, len(train_dl)*3)

for epoch in range(3):
    model.train()
    loss_sum = 0
    for batch in train_dl:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out = model(**batch)
        out.loss.backward()
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
        loss_sum += out.loss.item()
    print(f'Epoch {epoch+1} loss: {loss_sum/len(train_dl):.4f}')

In [ ]:
model.eval()
test_dl = DataLoader(NoteDS(test_rows, labeled=False), batch_size=16)
preds = []
with torch.no_grad():
    for batch in test_dl:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        preds.extend(model(**batch).logits.argmax(-1).cpu().tolist())

with open('predictions.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['note_id','predicted_condition'])
    w.writeheader()
    for row, p in zip(test_rows, preds):
        w.writerow({'note_id': row['note_id'], 'predicted_condition': ID2LABEL[p]})

print(f'Saved {len(preds)} predictions')